# 03 · Evaluate and compare all models: FF++ (in-dataset) vs. Celeb-DF (cross-dataset)

Every trained model is tested on:
- **FF++ test**: same kind of data it was trained on (held-out people)
- **Celeb-DF test**: an **unseen dataset** (different generator, people and quality). This measures generalisation.

For each model: AUC with **95% confidence intervals** (bootstrap over *videos*, since frames of one video aren't
independent), accuracy and F1 at the threshold tuned on FF++ validation (never re-tuned on a test set), video-level AUC,
AUC per manipulation method, **parameters, GFLOPs, speed and training time**, and **paired significance tests** against
EfficientNet-B0.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 · **Internet:** On
- **Inputs:** **01_extract_faces** (FF++), **01b_extract_celebdf**, and **02_train** (plus any copies of 02)

Before evaluating, it **audits the data for leakage** (identity/video overlap between splits, duplicate images across splits and datasets).

**Produces:** `/kaggle/working/outputs/compare/` (CSV tables + figures for the paper).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Uses the repo if it is attached as an input (zip uploaded as a Kaggle dataset, or an earlier
# notebook's output); otherwise clones it. For a private repo add a GITHUB_TOKEN secret.
import os, sys, glob, shutil, subprocess

PROJECT = "/kaggle/working/project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
    if found:
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
        if r.returncode:
            raise RuntimeError("git clone failed: attach the project as a dataset or add a GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
print("project ready:", sorted(os.listdir(PROJECT)))

In [ ]:
from src.kaggle_utils import setup, find_datasets, checkpoint_roots
setup(data=False)

import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Image as Show

DATASETS = find_datasets()
ROOTS = checkpoint_roots()
print("datasets:", DATASETS)
print("checkpoint folders:", ROOTS)
assert "FF++" in DATASETS, "Add the output of 01_extract_faces as an input"
assert ROOTS, "Add the output of 02_train as an input"
if "CelebDF" not in DATASETS:
    print("WARNING: Celeb-DF crops not attached; only the FF++ evaluation will run.")

## Data-leakage audit
Runs on the actual face crops before any model is evaluated. **FAIL** = leakage (stop and fix before trusting results); **WARN** = worth a look.
- every image on disk is in the manifest (no strays from earlier runs)
- no video and no FF++ identity appears in more than one split
- no identical image in two splits or in both FF++ and Celeb-DF; near-duplicates via perceptual hash
- Celeb-DF is test-only; class balance per split

In [ ]:
args = f'--data "{DATASETS["FF++"]}" --workers 4'
if "CelebDF" in DATASETS: args += f' --celebdf "{DATASETS["CelebDF"]}"'
!TQDM_MININTERVAL=60 python -m src.classifier.check_leakage {args}

## Run the comparison
About 1–2 minutes per model per dataset, plus the bootstrap.

In [ ]:
roots = " ".join(f'"{r}"' for r in ROOTS)
datasets = " ".join(f'"{k}={v}"' for k, v in DATASETS.items() if k in ("FF++", "CelebDF"))
!TQDM_MININTERVAL=60 python -m src.classifier.compare --roots {roots} --datasets {datasets} --bootstrap 1000 --workers 4
OUT = "/kaggle/working/outputs/compare"

## Main table
`*_auc_ci` = 95% CI from the video bootstrap. `auc_drop` = FF++ AUC − Celeb-DF AUC (lower = generalises better).

In [ ]:
summary = pd.read_csv(f"{OUT}/summary.csv")
display(summary.round(3))
for fig in ("generalisation.png", "auc_vs_compute.png"):
    if os.path.exists(f"{OUT}/{fig}"): display(Show(f"{OUT}/{fig}", width=650))

## Per dataset: metrics, significance vs. EfficientNet-B0, per-method AUC, ROC

In [ ]:
for ds in [d for d in ("FF++", "CelebDF") if os.path.isdir(f"{OUT}/{d}")]:
    display(Markdown(f"### {ds}"))
    display(pd.read_csv(f"{OUT}/{ds}/comparison.csv").drop(columns="backbone").round(4))
    sig = f"{OUT}/{ds}/significance.csv"
    if os.path.getsize(sig) > 1:
        display(Markdown("**Paired bootstrap: EfficientNet-B0 minus other model** (CI excluding 0 / p < 0.05 = significant)"))
        display(pd.read_csv(sig).round(4))
    display(pd.read_csv(f"{OUT}/{ds}/per_method.csv", index_col=0).round(4))
    display(Show(f"{OUT}/{ds}/roc_all.png", width=450))

## Celeb-DF examples: where EfficientNet-B0 is most wrong

In [ ]:
pred_csv = f"{OUT}/CelebDF/predictions.csv"
if os.path.exists(pred_csv):
    p = pd.read_csv(pred_csv)
    worst = pd.concat([p[p.label == 0].nlargest(6, "efficientnet_b0"), p[p.label == 1].nsmallest(6, "efficientnet_b0")])
    fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
    for ax, r in zip(axes.flat, worst.itertuples()):
        cls = "real" if r.label == 0 else "fake"
        ax.imshow(Image.open(f"{DATASETS['CelebDF']}/test/{cls}/{r.file}")); ax.axis("off")
        ax.set_title(f"{cls} · P(fake)={r.efficientnet_b0:.2f}", fontsize=8)
    plt.suptitle("Top: real Celeb-DF frames with the highest fake score · Bottom: fakes with the lowest"); plt.tight_layout(); plt.show()